# 02 — Read, join, and report

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 1, Lessons 3, 5, and 6.

Goal: Inspect raw CSV fields, preserve guest sales, and produce a first sales report.

Before running: Run 00 successfully. Teach notebook 03 between the reading and joining sections as Lesson 4.

Writes: No persistent tables; this notebook explores the canonical clean batch.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## Supplied support cells

These utilities keep repeated setup and checks in one place inside this notebook. The Python cleaner deliberately processes this tiny fixture on the driver so beginners can inspect each row. Spark handles DataFrames, joins, summaries, and Delta tables. This is a teaching pattern for 80 sales, not a large-scale ingestion design. The helper definitions below are part of the notebook and can be inspected. Learners will construct selected parts through Build Buddy in the later authoring phase.


In [0]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date, datetime
from collections import Counter, defaultdict
import csv, json, re

def read_rows(relative_path):
    with (Path(DATASET_ROOT) / relative_path).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def money(value):
    return Decimal(value).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

def business_payload(row):
    return {key: value for key, value in row.items() if key != 'source_row_id'}

def business_records(rows):
    return sorted([business_payload(row) for row in rows], key=lambda row: row['sale_id'])

def dimension_maps():
    return (
        {r['store_id']: r for r in read_rows('data/stores.csv')},
        {r['product_id']: r for r in read_rows('data/products.csv')},
        {r['customer_id']: r for r in read_rows('data/customers.csv')},
    )

def clean_sales(raw_rows):
    """Readable Python cleaning for this tiny fixture; no distributed UDF required."""
    stores, products, customers = dimension_maps()
    candidates, rejected, repaired = [], [], []
    for raw in sorted(raw_rows, key=lambda row: row['source_row_id']):
        row = {key: value.strip() for key, value in raw.items()}
        for key in ['store_id', 'product_id', 'customer_id', 'payment_method']:
            row[key] = row[key].upper()
        if row['customer_id'] == 'UNKNOWN':
            row['customer_id'] = ''
        reason = None
        if any(not row[k] for k in ['sale_id', 'source_row_id', 'sale_date', 'store_id', 'product_id']):
            reason = 'REQUIRED_MISSING'
        if not reason:
            try:
                fmt = '%d/%m/%Y' if '/' in row['sale_date'] else '%Y-%m-%d'
                pattern = r'\d{2}/\d{2}/\d{4}' if '/' in row['sale_date'] else r'\d{4}-\d{2}-\d{2}'
                if not re.fullmatch(pattern, row['sale_date']):
                    raise ValueError('Date format mismatch')
                row['sale_date'] = datetime.strptime(row['sale_date'], fmt).date().isoformat()
            except ValueError:
                reason = 'DATE_INVALID'
        if not reason:
            if row['product_id'] not in products:
                reason = 'PRODUCT_UNKNOWN'
            elif row['store_id'] not in stores:
                reason = 'STORE_UNKNOWN'
            elif row['customer_id'] and row['customer_id'] not in customers:
                reason = 'CUSTOMER_UNKNOWN'
        if not reason:
            try:
                if not re.fullmatch(r'\d+', row['quantity']) or int(row['quantity']) < 1:
                    reason = 'QUANTITY_INVALID'
                elif not re.fullmatch(r'\d+', row['record_version']) or int(row['record_version']) < 1:
                    reason = 'VERSION_INVALID'
                else:
                    row['unit_price'] = format(money(row['unit_price'] or products[row['product_id']]['list_price']), '.2f')
                    row['discount_pct'] = row['discount_pct'] or '0'
                    price, discount = Decimal(row['unit_price']), Decimal(row['discount_pct'])
                    if not price.is_finite() or price <= 0:
                        reason = 'PRICE_INVALID'
                    elif not discount.is_finite() or not 0 <= discount <= 100:
                        reason = 'DISCOUNT_INVALID'
                    elif row['payment_method'] not in ['UPI', 'CARD', 'CASH']:
                        reason = 'PAYMENT_INVALID'
            except (ValueError, InvalidOperation):
                reason = 'NUMBER_INVALID'
        if reason:
            rejected.append(dict(raw, rejection_reason=reason))
        else:
            if row != raw:
                repaired.append(row['source_row_id'])
            candidates.append(row)
    groups = defaultdict(list)
    for row in candidates:
        groups[(row['sale_id'], row['record_version'])].append(row)
    accepted, duplicate_ids = [], []
    raw_by_id = {row['source_row_id']: row for row in raw_rows}
    for group in groups.values():
        if any(business_payload(row) != business_payload(group[0]) for row in group):
            rejected.extend(dict(raw_by_id[row['source_row_id']], rejection_reason='VERSION_CONFLICT') for row in group)
        else:
            accepted.append(group[0])
            duplicate_ids.extend(row['source_row_id'] for row in group[1:])
    return accepted, rejected, duplicate_ids, repaired

def current_versions(rows):
    state = {}
    for row in rows:
        old = state.get(row['sale_id'])
        if old is None or int(row['record_version']) > int(old['record_version']):
            state[row['sale_id']] = row
    return sorted(state.values(), key=lambda row: row['sale_id'])

def visible_returns(stage, sales_rows):
    rows = [r for r in read_rows('data/returns.csv') if stage == 'final' or r['available_batch'] == '01']
    sales = {row['sale_id']: row for row in sales_rows}
    assert len({r['return_id'] for r in rows}) == len(rows)
    returned = Counter()
    for row in rows:
        sale = sales[row['sale_id']]
        assert row['available_batch'] in ['01', '02']
        assert date.fromisoformat(row['return_date']) >= date.fromisoformat(sale['sale_date'])
        quantity = int(row['return_quantity'])
        assert quantity > 0
        returned[row['sale_id']] += quantity
        assert returned[row['sale_id']] <= int(sale['quantity'])
        assert Decimal(row['refund_amount']) == money(quantity * Decimal(sale['unit_price']) * (1 - Decimal(sale['discount_pct']) / 100))
    return rows


In [0]:
from pyspark.sql import functions as F, types as T

SALES_COLUMNS = ['source_row_id', 'sale_id', 'sale_date', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'record_version']
SALES_SCHEMA = T.StructType([
    T.StructField(name, T.DateType() if name == 'sale_date' else
                  T.IntegerType() if name in ['quantity', 'record_version'] else
                  T.DecimalType(12, 2) if name == 'unit_price' else
                  T.DecimalType(5, 2) if name == 'discount_pct' else T.StringType(), True)
    for name in SALES_COLUMNS
])

def sales_frame(rows):
    typed = []
    for row in rows:
        values = []
        for name in SALES_COLUMNS:
            value = row[name]
            if name == 'sale_date': value = date.fromisoformat(value)
            elif name in ['quantity', 'record_version']: value = int(value)
            elif name in ['unit_price', 'discount_pct']: value = Decimal(value)
            elif name == 'customer_id' and value == '': value = None
            values.append(value)
        typed.append(tuple(values))
    return spark.createDataFrame(typed, SALES_SCHEMA)

def raw_frame(relative_path):
    return spark.read.option('header', True).option('inferSchema', False).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/{relative_path}')

def table_name(suffix):
    assert re.fullmatch(r'[a-z][a-z0-9_]*', suffix)
    return f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{suffix}'

def assert_expected(frame, relative_path, ignore_provenance=False, frame_schema=None):
    """Compare a frame against checks/expected/<relative_path>.
    Spark Connect: analyze the schema once in the calling cell and pass frame_schema in."""
    if frame_schema is None:
        raise ValueError('Pass frame_schema=<frame>.schema from the calling cell.')
    columns = list(frame_schema.fieldNames())
    if ignore_provenance and 'source_row_id' in columns:
        columns.remove('source_row_id')
    actual = frame.select(*columns)
    # Read full files before dropping provenance so schema fields match CSV positions.
    expected = spark.read.csv(f'{DATASET_ROOT}/checks/expected/{relative_path}', schema=frame_schema, header=True, mode='FAILFAST').select(*columns)
    assert actual.exceptAll(expected).limit(1).count() == 0, f'Extra/different rows: {relative_path}'
    assert expected.exceptAll(actual).limit(1).count() == 0, f'Missing rows: {relative_path}'
    print('PASS:', relative_path)

def collected_sales(frame):
    rows = []
    for record in frame.select(*SALES_COLUMNS).collect():
        row = record.asDict()
        row['sale_date'] = row['sale_date'].isoformat()
        row['quantity'] = str(row['quantity'])
        row['record_version'] = str(row['record_version'])
        row['unit_price'] = format(row['unit_price'], '.2f')
        row['discount_pct'] = format(row['discount_pct'].normalize(), 'f')
        row['customer_id'] = row['customer_id'] or ''
        rows.append(row)
    return sorted(rows, key=lambda row: row['sale_id'])

def merge_counts(current, incoming):
    joined = incoming.alias('s').join(current.alias('t'), F.col('s.sale_id') == F.col('t.sale_id'), 'left')
    business_columns = [c for c in SALES_COLUMNS if c != 'source_row_id']
    equal = F.lit(True)
    for name in business_columns:
        equal = equal & F.col(f's.{name}').eqNullSafe(F.col(f't.{name}'))
    conflicts = joined.where((F.col('s.record_version') == F.col('t.record_version')) & ~equal)
    assert conflicts.limit(1).count() == 0, 'Equal-version business conflict: investigate before merge.'
    counts = {'insert': 0, 'update': 0, 'noop': 0}
    for row in joined.select(
        F.when(F.col('t.sale_id').isNull(), 'insert')
         .when(F.col('s.record_version') > F.col('t.record_version'), 'update')
         .otherwise('noop').alias('action')
    ).groupBy('action').count().collect():
        counts[row['action']] = row['count']
    return counts

def merge_into_silver(incoming):
    from delta.tables import DeltaTable
    assert incoming.groupBy('sale_id').count().where('count > 1').limit(1).count() == 0
    target = table_name('silver_sales')
    counts = merge_counts(spark.table(target), incoming)
    (DeltaTable.forName(spark, target).alias('t')
        .merge(incoming.alias('s'), 't.sale_id = s.sale_id')
        .whenMatchedUpdateAll(condition='s.record_version > t.record_version')
        .whenNotMatchedInsertAll().execute())
    return counts


## 02-01: Read rows as strings

**Core concept Vibe:** A schema describes column names and types.

**Build Buddy action:** Read initial sales without automatic type guessing and inspect the first records.

**Expected checkpoint:** 60 rows; all raw fields are strings. Stop here to teach notebook 03 for Lesson 4.


In [0]:
# Build Buddy step 02-01
# Read the initial sales file with no automatic type guessing: every field stays a string.
raw_rows = read_rows('data/sales_batch_01.csv')
print('Raw rows:', len(raw_rows))
print('First record:', raw_rows[0])
assert len(raw_rows) == 60
assert all(isinstance(value, str) for row in raw_rows for value in row.values())
print('CHECKPOINT 02-01 PASS: 60 rows; all raw fields are strings')


Raw rows: 60
First record: {'source_row_id': 'B01-001', 'sale_id': 'S0001', 'sale_date': '2026-01-01', 'store_id': 'ST01', 'product_id': 'P001', 'customer_id': 'C001', 'quantity': '1', 'unit_price': '80.00', 'discount_pct': '0', 'payment_method': 'UPI', 'record_version': '1'}
CHECKPOINT 02-01 PASS: 60 rows; all raw fields are strings


## 02-02: Cast checked values

**Core concept Vibe:** Validate before turning text into dates or numbers.

**Build Buddy action:** Validate initial sales and create a typed Spark DataFrame.

**Expected checkpoint:** 60 typed sales; 10 guest sales remain valid.


In [0]:
# Build Buddy step 02-02
# Validate the checked text, then cast it into typed columns.
accepted01, rejected01, duplicate_ids01, repaired01 = clean_sales(raw_rows)
print('Accepted:', len(accepted01), '| rejected:', len(rejected01), '| repaired:', len(repaired01))
sales = sales_frame(accepted01)
guest_sales = sales.where('customer_id IS NULL').count()
print('Typed sales:', sales.count(), '| guest sales:', guest_sales)
assert sales.count() == 60 and len(rejected01) == 0 and guest_sales == 10
print('CHECKPOINT 02-02 PASS: 60 typed sales; 10 guest sales remain valid')


Accepted: 60 | rejected: 0 | repaired: 0
Typed sales: 60 | guest sales: 10
CHECKPOINT 02-02 PASS: 60 typed sales; 10 guest sales remain valid


## 02-03: Join by identifiers

**Core concept Vibe:** A left join preserves every sale, including guests.

**Build Buddy action:** Enrich sales with store, product and customer names.

**Expected checkpoint:** 60 enriched sales; S0001 resolves to Asha; S0006 remains present without a customer name.


In [0]:
# Build Buddy step 02-03
# A left join preserves every sale, including guest sales.
stores = raw_frame('data/stores.csv').select('store_id', 'store_name')
products = raw_frame('data/products.csv').select('product_id', 'product_name')
customers = raw_frame('data/customers.csv').select('customer_id', 'customer_name')
enriched = (sales
    .join(stores, 'store_id', 'left')
    .join(products, 'product_id', 'left')
    .join(customers, 'customer_id', 'left'))
s0001 = enriched.where("sale_id = 'S0001'").select('sale_id', 'store_name', 'product_name', 'customer_name').first().asDict()
s0006 = enriched.where("sale_id = 'S0006'").select('sale_id', 'customer_id', 'customer_name').first().asDict()
print('Enriched sales:', enriched.count())
print('S0001:', s0001)
print('S0006:', s0006)
assert enriched.count() == 60
assert s0001['customer_name'] == 'Asha'
assert s0006['customer_id'] is None and s0006['customer_name'] is None
print('CHECKPOINT 02-03 PASS: 60 enriched sales; S0001 resolves to Asha; S0006 stays present without a customer name')


Enriched sales: 60
S0001: {'sale_id': 'S0001', 'store_name': 'RetailFlow Mumbai', 'product_name': 'Rice 1 kg', 'customer_name': 'Asha'}
S0006: {'sale_id': 'S0006', 'customer_id': None, 'customer_name': None}
CHECKPOINT 02-03 PASS: 60 enriched sales; S0001 resolves to Asha; S0006 stays present without a customer name


## 02-04: Calculate and group sales

**Core concept Vibe:** Grouping combines records that share a key.

**Build Buddy action:** Calculate discounted sales and summarize them by store.

**Expected checkpoint:** Total net sales before returns = INR 14,456.00; every store agrees with the answer key.


In [0]:
# Build Buddy step 02-04
# Discounted amount per sale, then grouped by store.
store_lines = sales.select(
    'store_id', 'sale_id', 'quantity',
    (F.col('quantity') * F.col('unit_price')).cast('decimal(12,2)').alias('gross_sales'),
    F.round((F.col('quantity') * F.col('unit_price')).cast('decimal(12,2)') * F.col('discount_pct') / F.lit(100), 2).cast('decimal(12,2)').alias('discount_amount'),
).withColumns({'net_sales': (F.col('gross_sales') - F.col('discount_amount')).cast('decimal(12,2)')})
by_store = store_lines.groupBy('store_id').agg(
    F.count('sale_id').alias('sale_count'),
    F.sum('quantity').alias('units_sold'),
    F.sum('net_sales').alias('net_sales'),
).orderBy('store_id')
by_store.show(truncate=False)
total_net_sales = by_store.groupBy().agg(F.sum('net_sales').alias('total')).first()['total']
print('Total net sales before returns:', total_net_sales)
assert total_net_sales == Decimal('14456.00')
expected_by_store = {row['store_id']: Decimal(row['net_sales']) for row in read_rows('checks/expected/initial_by_store.csv')}
computed_by_store = {row['store_id']: row['net_sales'] for row in by_store.collect()}
assert computed_by_store == expected_by_store, (computed_by_store, expected_by_store)
print('CHECKPOINT 02-04 PASS: total net sales before returns = INR 14,456.00; every store agrees with the answer key')


+--------+----------+----------+---------+
|store_id|sale_count|units_sold|net_sales|
+--------+----------+----------+---------+
|ST01    |6         |12        |1858.00  |
|ST02    |6         |12        |1995.00  |
|ST03    |6         |12        |1923.00  |
|ST04    |6         |12        |1584.00  |
|ST05    |6         |12        |1008.00  |
|ST06    |6         |12        |948.00   |
|ST07    |6         |12        |936.00   |
|ST08    |6         |12        |997.00   |
|ST09    |6         |12        |1398.00  |
|ST10    |6         |12        |1809.00  |
+--------+----------+----------+---------+

Total net sales before returns: 14456.00
CHECKPOINT 02-04 PASS: total net sales before returns = INR 14,456.00; every store agrees with the answer key


## 02-05: Show products with no sales

**Core concept Vibe:** Starting from the product list preserves unsold products.

**Build Buddy action:** Left join product totals onto all 12 products.

**Expected checkpoint:** P012 Desk Lamp appears with zero units sold.


In [0]:
# Build Buddy step 02-05
# Start from the full product list so unsold products remain visible.
product_lines = sales.select(
    'product_id', 'sale_id', 'quantity',
    (F.col('quantity') * F.col('unit_price')).cast('decimal(12,2)').alias('gross_sales'),
    F.round((F.col('quantity') * F.col('unit_price')).cast('decimal(12,2)') * F.col('discount_pct') / F.lit(100), 2).cast('decimal(12,2)').alias('discount_amount'),
).withColumns({'net_sales': (F.col('gross_sales') - F.col('discount_amount')).cast('decimal(12,2)')})
product_totals = product_lines.groupBy('product_id').agg(
    F.count('sale_id').alias('sale_count'),
    F.sum('quantity').alias('units_sold'),
    F.sum('net_sales').alias('net_sales'),
)
all_products = raw_frame('data/products.csv').select('product_id', 'product_name')
by_product = (all_products.join(product_totals, 'product_id', 'left').fillna(0)
    .select('product_id', 'product_name', 'sale_count', 'units_sold', 'net_sales')
    .orderBy('product_id'))
by_product.show(truncate=False)
p012 = by_product.where("product_id = 'P012'").first().asDict()
print('P012 row:', p012)
assert by_product.count() == 12
assert p012['product_name'] == 'Desk Lamp' and p012['sale_count'] == 0 and p012['units_sold'] == 0
print('CHECKPOINT 02-05 PASS: all 12 products listed; P012 Desk Lamp appears with zero units sold')


+----------+--------------+----------+----------+---------+
|product_id|product_name  |sale_count|units_sold|net_sales|
+----------+--------------+----------+----------+---------+
|P001      |Rice 1 kg     |6         |12        |896.00   |
|P002      |Milk 1 L      |6         |12        |672.00   |
|P003      |Tea 250 g     |6         |12        |1680.00  |
|P004      |Soap 100 g    |6         |12        |448.00   |
|P005      |Shampoo 200 mL|6         |12        |1356.00  |
|P006      |Notebook A5   |5         |11        |510.00   |
|P007      |Pen Pack      |5         |10        |279.00   |
|P008      |Water Bottle  |5         |9         |1700.00  |
|P009      |Lunch Box     |5         |11        |3060.00  |
|P010      |Towel         |5         |10        |2325.00  |
|P011      |Coffee 100 g  |5         |9         |1530.00  |
|P012      |Desk Lamp     |0         |0         |0.00     |
+----------+--------------+----------+----------+---------+

P012 row: {'product_id': 'P012', 'produ